In [0]:
from pyspark.sql import SparkSession


In [0]:
'''storage_account_name = "nyctaxistoragegrp"
container_name = "tripdatacontainer"

# Retrieve storage key from Databricks Secret Scope (never hardcode secrets)
storage_account_key = dbutils.secrets.get(scope="<your-secret-scope>", key="<your-storage-key-secret>")

# Set the Spark configuration
spark.conf.set("fs.azure.account.key.nyctaxistoragegrp.blob.core.windows.net", 
    storage_account_key)

# Read a CSV file
file_path = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/*.parquet"

df = spark.read.format("parquet") \
    .option("header", "true") \
    .load(file_path)
'''

In [0]:
# Read Entire dataSet
df = spark.read.parquet("/Volumes/workspace/taxi_project/2021")

In [0]:
# count rows
df.count()

In [0]:
# see coloumns
df.columns


In [0]:
# see Schema
df.printSchema()

In [0]:
# see Total files in the Volume
df.select("_metadata.file_path").distinct().show(truncate=False)

In [0]:
# alternate way to view all files
files = dbutils.fs.ls("/Volumes/workspace/taxi_project/2021")

for f in files:
    print(f.path)

In [0]:
# Check Shape
print("Rows :", df.count())
print("Columns :", len(df.columns))

In [0]:
# View DATA
display(df.limit(20))

In [0]:
# Check data Types
df.dtypes

In [0]:
# Dictionary type of datatypes of columns
dict(df.dtypes)

In [0]:
# Describe
display(df.describe())

In [0]:
# Display Summary of data
display(df.summary())

In [0]:
# Explain dataframe
df.explain()

In [0]:
# Missing Values
from pyspark.sql.functions import col, when, count

display(
    df.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in df.columns
    ])
)

In [0]:
# Null Rows
from pyspark.sql.functions import *

total = df.count()

display(
(
df.select([
(count(when(col(c).isNull(),c))/lit(total)*100).alias(c)
for c in df.columns
])
)
)

In [0]:
# fill with the mean
from pyspark.sql.functions import mean

numeric_cols = [
    c for c, t in df.dtypes
    if t in ("double", "float", "int", "bigint")
]

for c in numeric_cols:
    avg = df.select(mean(c)).first()[0]
    if avg is not None:
        df = df.fillna({c: avg})

In [0]:
# FIll with Unknown
string_cols = [
    c for c, t in df.dtypes
    if t == "string"
]

fill_dict = {c: "Unknown" for c in string_cols}

df = df.fillna(fill_dict)

In [0]:
# Drop Duplicates and then see
duplicates = df.count() - df.dropDuplicates().count()

print(duplicates)

df = df.dropDuplicates()
display(df.describe())

In [0]:
duplicates = df.count()-df.dropDuplicates().count()

In [0]:
# Unique Values in columns
for c in df.columns:
    print(c, df.select(c).distinct().count())

In [0]:
from pyspark.sql.functions import round, avg, sum, count,col

In [0]:
# Check both NULL and NaN
from pyspark.sql.types import DoubleType, FloatType
from pyspark.sql.functions import col, when, count, isnan

exprs = []

for field in df.schema.fields:
    if isinstance(field.dataType, (DoubleType, FloatType)):
        exprs.append(
            count(
                when(col(field.name).isNull() | isnan(col(field.name)), field.name)
            ).alias(field.name)
        )
    else:
        exprs.append(
            count(
                when(col(field.name).isNull(), field.name)
            ).alias(field.name)
        )

missing = df.select(exprs)

display(missing)

In [0]:
# Check Numeric Columns
numeric = [
    c for c,t in df.dtypes
    if t in ['int','bigint','double','float','decimal']
]

numeric

In [0]:
# Check Categorical Columns
categorical = [
    c for c,t in df.dtypes
    if t=="string"
]

categorical


# Invalid Records
tip_amount <0
total_amount<0
extra<0
mta_tax<0
tolls_amount<0
display(df.filter(col("fare_amount")<0))

In [0]:
# Value Counts

display(df.groupBy("VendorID").count())

In [0]:
# Correlation
display(df.select(numeric).summary())

In [0]:
# Outlier Detection for fare_amount
Q1,Q3 = df.approxQuantile("fare_amount",[0.25,0.75],0)

IQR = Q3-Q1

lower = Q1-1.5*IQR
upper = Q3+1.5*IQR

print(lower,upper)

In [0]:
# Find Outliers
display(
    df.filter(
        (col("fare_amount")<lower)|
        (col("fare_amount")>upper)
    )
)

In [0]:
# Remove them
df = df.filter(
    (col("fare_amount")>=lower)&
    (col("fare_amount")<=upper)
)

In [0]:
# This keeps all rows and replaces extreme values with the IQR bounds.
from pyspark.sql.functions import col, when

continuous_cols = [
    "fare_amount",
    "trip_distance",
    "total_amount",
    "tip_amount",
    "tolls_amount"
]

for c in continuous_cols:

    if c not in df.columns:
        print(f"{c} not found. Skipping...")
        continue

    Q1, Q3 = df.approxQuantile(c, [0.25, 0.75], 0)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df = df.withColumn(
        c,
        when(col(c) < lower, lower)
        .when(col(c) > upper, upper)
        .otherwise(col(c))
    )

print("Outlier capping completed.")

In [0]:
from pyspark.sql.functions import when

df = df.withColumn(
    "fare_amount",
    when(col("fare_amount")>upper,upper)
    .when(col("fare_amount")<lower,lower)
    .otherwise(col("fare_amount"))
)

In [0]:
# check for invalid values
display(df.filter(col("fare_amount")<0))

In [0]:
display(df.filter(col("trip_distance")<0))

In [0]:
display(df.filter(col("passenger_count")<=0))

In [0]:
display(
df.filter(
col("tpep_pickup_datetime")>
col("tpep_dropoff_datetime")
)
)

In [0]:
display(df.filter(col("trip_distance")==0))

In [0]:
# Feature Engineering
from pyspark.sql.functions import unix_timestamp

df = df.withColumn(
"trip_duration",
(unix_timestamp("tpep_dropoff_datetime")-
unix_timestamp("tpep_pickup_datetime"))/60
)

In [0]:
# Speed
from pyspark.sql.functions import *

df = df.withColumn(
    "speed",
    when(
        col("trip_duration") > 0,
        col("trip_distance") / col("trip_duration")
    ).otherwise(None)
)

In [0]:
# dates
from pyspark.sql.functions import *

df=df.withColumn("year",year("tpep_pickup_datetime"))\
.withColumn("month",month("tpep_pickup_datetime"))\
.withColumn("day",dayofmonth("tpep_pickup_datetime"))\
.withColumn("hour",hour("tpep_pickup_datetime"))

In [0]:
# WEEKEND
from pyspark.sql.functions import dayofweek

df=df.withColumn(
"weekend",
when(dayofweek("tpep_pickup_datetime").isin([1,7]),1).otherwise(0)
)

In [0]:
# Night Trip
df=df.withColumn(
"night_trip",
when(col("hour").between(20,23)|col("hour").between(0,5),1).otherwise(0)
)

In [0]:
# Fare per Mile
df=df.withColumn(
"fare_per_mile",
col("fare_amount")/col("trip_distance")
)

In [0]:
# Tip percentage 
df=df.withColumn(
"tip_percent",
(col("tip_amount")/col("fare_amount"))*100
)

In [0]:
df.printSchema()

In [0]:
print(df.columns)

In [0]:
# Visiualize the fare amount
display(
    df.select("fare_amount")
)

Databricks visualization. Run in Databricks to view.

In [0]:
# Top 10 Dropoff Locations
display(
    df.groupBy("DOLocationID")
      .count()
      .orderBy("count", ascending=False)
      .limit(10)
)

Databricks visualization. Run in Databricks to view.

In [0]:
display(
    df.groupBy("payment_type").count()
)

Databricks visualization. Run in Databricks to view.

# Business

In [0]:
# Revenue analysis
from pyspark.sql.functions import sum

df.select(
    sum("total_amount").alias("Total Revenue")
).show()

In [0]:
# revenue by month

from pyspark.sql.functions import sum

display(
    df.groupBy("month")
      .agg(round(sum("total_amount"),2).alias("Revenue"))
      .orderBy("month")
)

In [0]:
# Revenue by Vendor
display(
    df.groupBy("VendorID")
      .agg(round(sum("total_amount"),2).alias("Revenue"))
      .orderBy("Revenue", ascending=False)
)

In [0]:
# Trip analysis
display(
    df.groupBy("hour")
      .count()
      .orderBy("hour")
)

In [0]:
# Trip analysis by day of week
from pyspark.sql.functions import dayofweek

display(
    df.withColumn("weekday", dayofweek("tpep_pickup_datetime"))
      .groupBy("weekday")
      .count()
      .orderBy("weekday")
)

In [0]:
# Trip analysis by month
display(
    df.groupBy("month")
      .count()
      .orderBy("month")
)

In [0]:
# Passenger Analysis
from pyspark.sql.functions import round, col

display(
    df.groupBy(round(col("passenger_count"), 2).alias("passenger_count"))
      .count()
      .orderBy("passenger_count")
)

In [0]:
# Payment Analysis
from pyspark.sql.functions import sum

display(
    df.groupBy("payment_type")
      .agg(
          round(sum("total_amount"),2).alias("Revenue")
      )
      .orderBy("Revenue", ascending=False)
)

In [0]:
# Tip Analysis
from pyspark.sql.functions import avg

display(
    df.groupBy("payment_type")
      .agg(
          avg("tip_amount").alias("Average Tip")
      )
)

In [0]:
# Distance Analysis
display(
    df.select("trip_distance").summary()
)

In [0]:
# Trip Duration Analysis
display(
    df.select("trip_duration").summary()
)

In [0]:
# Speed Analysis
display(
    df.select("speed").summary()
)

In [0]:
# Pickup Hotspots
display(
    df.groupBy("PULocationID")
      .count()
      .orderBy("count", ascending=False)
      .limit(20)
)

In [0]:
# Dropoff Hotspots
display(
    df.groupBy("DOLocationID")
      .count()
      .orderBy("count", ascending=False)
      .limit(20)
)

In [0]:
# Airport analysis
display(
    df.groupBy("Airport_fee")
      .count()
)


In [0]:
# revenue from airport analysis
from pyspark.sql.functions import sum

display(
    df.groupBy("Airport_fee")
      .agg(round(sum("total_amount"),2).alias("Revenue"))
)

In [0]:
# Peak Hour Revenue
from pyspark.sql.functions import sum

display(
    df.groupBy("hour")
      .agg(round(sum("total_amount"),2).alias("Revenue"))
      .orderBy("hour")
)


In [0]:
# Average Fare by Hour
from pyspark.sql.functions import avg

display(
    df.groupBy("hour")
      .agg(avg("fare_amount").alias("Average Fare"))
      .orderBy("hour")
)

In [0]:
# Average Fare by Vendor

display(
    df.groupBy("VendorID")
      .agg(
          round(avg("fare_amount"), 2).alias("Average Fare")
      )
)

In [0]:
# Revenue per kilometer
from pyspark.sql.functions import sum

display(
    df.groupBy("VendorID")
      .agg(
          round((
              sum("total_amount") /
              sum("trip_distance")
          ),2).alias("Revenue_per_KM")
      )
)

In [0]:
# Highest Revenue day
display(
df.groupBy("day")
.agg(round(sum("total_amount"),2).alias("Revenue"))
.orderBy("Revenue",ascending=False)
)

In [0]:
# Average Revenue per trip
from pyspark.sql.functions import avg

df.select(
avg("total_amount")
.alias("Average Revenue")
).show()

In [0]:
# Revenue  by Passenger Count
display(
df.groupBy("passenger_count")
.agg(round(sum("total_amount"),2)
))

In [0]:
# Longest Average Trips
display(
df.groupBy("PULocationID")
.agg(round(avg("trip_duration"),2)
)

In [0]:
# Correlation Analysis -This shows whether longer trips produce higher fares.
print(
    df.stat.corr(
        "trip_distance",
        "total_amount"
    )
)

print(
    df.stat.corr(
        "trip_duration",
        "total_amount"
    )
)

print(
    df.stat.corr(
        "trip_distance",
        "tip_amount"
    )
)

In [0]:

display(
spark.table(
"workspace.taxi_project.yellow_trip_clean"
)
)